In [1]:
from datasets import load_dataset

dataset = load_dataset("cais/mmlu", "all")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

all/auxiliary_train-00000-of-00001.parqu(…):   0%|          | 0.00/47.5M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

In [2]:
test_data = dataset["test"]
len(test_data)

14042

In [3]:
test_data.column_names

['question', 'subject', 'choices', 'answer']

In [4]:
import pandas as pd

df = test_data.to_pandas()
df.head()

,question,subject,choices,answer
0,Find the degree for the given field extension ...,abstract_algebra,"[0, 4, 2, 6]",1
1,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...",abstract_algebra,"[8, 2, 24, 120]",2
2,Find all zeros in the indicated finite field o...,abstract_algebra,"[0, 1, 0,1, 0,4]",3
3,Statement 1 | A factor group of a non-Abelian ...,abstract_algebra,"[True, True, False, False, True, False, False,...",1
4,Find the product of the given polynomials in t...,abstract_algebra,"[2x^2 + 5, 6x^2 + 4x + 6, 0, x^2 + 1]",1


In [5]:
subject_counts = df["subject"].value_counts()
subject_counts.head()

,count
subject,
professional_law,1534
moral_scenarios,895
miscellaneous,783
professional_psychology,612
high_school_psychology,545


In [6]:
total_needed = 604.5
proportions = subject_counts / subject_counts.sum()
samples_per_subject = (proportions * total_needed).round().astype(int)

In [7]:
samples_per_subject.sum()

np.int64(601)

Perform Stratified Sampling


In [8]:
import numpy as np

sampled_dfs = []

for subject, n_samples in samples_per_subject.items():
    subject_df = df[df["subject"] == subject]
    sampled_subject = subject_df.sample(n=n_samples, random_state=42)
    sampled_dfs.append(sampled_subject)

sampled_df = pd.concat(sampled_dfs).reset_index(drop=True)

len(sampled_df)

601

In [9]:
sampled_df = sampled_df.sample(frac=1, random_state=42).reset_index(drop=True)
sampled_df["split_type"] = None

sampled_df.loc[:199, "split_type"] = "verbatim"
sampled_df.loc[200:399, "split_type"] = "clean"
sampled_df.loc[400:599, "split_type"] = "paraphrase_pool"

In [10]:
sampled_df["split_type"].value_counts()

,count
split_type,
verbatim,200
clean,200
paraphrase_pool,200


In [ ]:
sampled_df.to_json("mmlu_sampled_600.json", orient="records", lines=True)

In [14]:
sampled_df.groupby(["split_type", "subject"]).size().head(20)

split_type  subject                            
clean       abstract_algebra                       2
            anatomy                                4
            astronomy                              3
            clinical_knowledge                     5
            college_mathematics                    1
            college_medicine                       2
            college_physics                        1
            computer_security                      4
            conceptual_physics                     1
            electrical_engineering                 5
            elementary_mathematics                 3
            formal_logic                           2
            global_facts                           1
            high_school_biology                    3
            high_school_chemistry                  3
            high_school_computer_science           2
            high_school_european_history           1
            high_school_geography                  2
            high_school_government_and_politics    4
            high_school_macroeconomics             6
dtype: int64

In [1]:
!pip install transformers sentence-transformers

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def paraphrase_text(text):
    prompt = f"Paraphrase the following question and choices while preserving meaning:\n{text}"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.8
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [4]:
def paraphrase_example(question, choices):

    formatted_input = f"""
Paraphrase the following multiple-choice question while preserving meaning.
Do NOT change the correct answer.
Keep number of options same.

Question:
{question}

Choices:
A. {choices[0]}
B. {choices[1]}
C. {choices[2]}
D. {choices[3]}

Return in this format exactly:

Question: <paraphrased question>
A. <choice>
B. <choice>
C. <choice>
D. <choice>
"""

    inputs = tokenizer(formatted_input, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.8
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [8]:
import pandas as pd

df = pd.read_json("mmlu_sampled_600.json", lines=True)

paraphrase_df = df[df["split_type"] == "paraphrase_pool"].copy()
paraphrase_df.reset_index(drop=True, inplace=True)

print(len(paraphrase_df))  # should be 200

200


In [9]:
!pip install sentence-transformers

In [13]:
from sentence_transformers import SentenceTransformer, util

embed_model = SentenceTransformer("all-mpnet-base-v2")

def similarity_score(original, new):
    emb1 = embed_model.encode(original, convert_to_tensor=True)
    emb2 = embed_model.encode(new, convert_to_tensor=True)
    return float(util.cos_sim(emb1, emb2))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
!pip install transformers sentence-transformers

In [16]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [23]:
def paraphrase_example(question, choices):

    text = question + " " + " ".join(choices)

    prompt = (
        "Rewrite the following multiple-choice question. "
        "Do not explain. Do not solve. "
        "Only rewrite the question and options with the same meaning.\n\n"
        + text
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        num_beams=4
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [28]:
import pandas as pd

df = pd.read_json("/mmlu_sampled_600.json", lines=True)

paraphrase_df = df[df["split_type"] == "paraphrase_pool"].copy()
paraphrase_df.reset_index(drop=True, inplace=True)

print("Paraphrase pool size:", len(paraphrase_df))

Paraphrase pool size: 200


In [29]:
row = paraphrase_df.iloc[0]

output = paraphrase_example(row["question"], row["choices"])

print(output)

A formally valid syllogism must not be materially true because _ . The formal validity of a syllogism does not guarantee it is materially true. The final answer: A formally valid syllogism.


In [30]:
original_text = row["question"] + " " + " ".join(row["choices"])

sim = similarity_score(original_text, output)

print("Similarity:", sim)

Similarity: 0.8769277334213257


In [32]:
paraphrased_records = []

for idx, row in paraphrase_df.iterrows():

    original_text = row["question"] + " " + " ".join(row["choices"])

    output = paraphrase_example(row["question"], row["choices"])

    sim = similarity_score(original_text, output)

    if sim > 0.65:
        paraphrased_records.append({
            "subject": row["subject"],
            "question": output,
            "choices": row["choices"],
            "answer": row["answer"],
            "split_type": "paraphrased"
        })
    else:
        print(f"Low similarity at index {idx}: {sim}")

Low similarity at index 7: 0.5798919200897217
Low similarity at index 8: 0.09653671830892563
Low similarity at index 10: 0.5506801605224609
Low similarity at index 11: 0.6404178142547607
Low similarity at index 12: 0.31127220392227173
Low similarity at index 14: 0.49676570296287537
Low similarity at index 15: 0.1497192680835724
Low similarity at index 17: 0.4995461106300354
Low similarity at index 18: 0.44346046447753906
Low similarity at index 22: 0.5721516013145447
Low similarity at index 23: 0.6287125945091248
Low similarity at index 24: 0.6291089057922363
Low similarity at index 26: 0.43904656171798706
Low similarity at index 27: 0.5694429874420166
Low similarity at index 28: 0.19797542691230774
Low similarity at index 33: 0.3693144917488098
Low similarity at index 35: 0.5474305152893066
Low similarity at index 36: 0.6355129480361938
Low similarity at index 37: 0.5184036493301392
Low similarity at index 40: 0.6282578110694885
Low similarity at index 41: 0.257744699716568
Low simila

In [33]:
print("Final paraphrased count:", len(paraphrased_records))

Final paraphrased count: 115


In [34]:
paraphrased_df = pd.DataFrame(paraphrased_records)

print(paraphrased_df.head())

                  subject                                           question  \
0       logical_fallacies  A formally valid syllogism must not be materia...   
1   high_school_chemistry  Question: A student has a liter of a 0.100 M s...   
2           miscellaneous  Who lost to Ronald Reagan in the 1984 presiden...   
3  high_school_psychology  Which of the following is not a basic somatose...   
4   high_school_chemistry  Which of the following is a reduction half-rea...   

                                             choices  answer   split_type  
0  [A formally valid syllogism may be materially ...       1  paraphrased  
1  [a strong acid, a weak acid, a weak base, a st...       2  paraphrased  
2  [Michael Dukakis, Walter Mondale, Gary Hart, J...       1  paraphrased  
3                          [pain, touch, cold, itch]       3  paraphrased  
4  [1 only because copper(II) ions are reduced, 3...       2  paraphrased  


In [37]:
paraphrased_df.to_json("mmlu_paraphrased_final.json", orient="records", lines=True)

In [38]:
verbatim_df = df[df["split_type"] == "verbatim"]
clean_df = df[df["split_type"] == "clean"]

print(len(verbatim_df), len(clean_df), len(paraphrased_df))

200 200 115


In [39]:
verbatim_df.to_json("mmlu_verbatim.json", orient="records", lines=True)
clean_df.to_json("mmlu_clean.json", orient="records", lines=True)